In [1]:
import os
import wfdb
import pandas as pd

In [2]:
input_path = '../data/'
output_path = '../mitbih_database/'

os.makedirs(output_path, exist_ok=True)

In [3]:
record_names = sorted([
    os.path.splitext(f)[0]
    for f in os.listdir(input_path)
    if f.endswith('.hea')
])

In [4]:
print(f'Liczba rekordów: {len(record_names)}')
print(f'Rekordy: {record_names}')

Liczba rekordów: 48
Rekordy: ['100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '111', '112', '113', '114', '115', '116', '117', '118', '119', '121', '122', '123', '124', '200', '201', '202', '203', '205', '207', '208', '209', '210', '212', '213', '214', '215', '217', '219', '220', '221', '222', '223', '228', '230', '231', '232', '233', '234']


In [5]:
aami_mapping = {
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N', # uderzenia przypisywane do N
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S', # uderzenia przypisywane do S
    'V': 'V', 'E': 'V', # uderzenia przypisywane do V
    'F': 'F', # uderzenia przypisywane do F
    '/': 'Q', 'f': 'Q', 'Q': 'Q' # uderzenia przypisywane do Q
}

In [6]:
for record_name in record_names:
    record_path = os.path.join(input_path, record_name)
    
    # zapisanie wartości sygnału do csv
    record = wfdb.rdrecord(record_path)
    
    ecg = pd.DataFrame(
        record.p_signal,
        columns=record.sig_name
    )
    ecg.insert(0, 'sample #', range(len(ecg)))
    
    ecg_path = os.path.join(output_path, f'{record_name}_record.csv')
    ecg.to_csv(ecg_path, index=False)
    
    # zapisanie adnotacji do drugiego csv
    annotation = wfdb.rdann(record_path, 'atr')
    
    annotations = pd.DataFrame({
        'Sample #': annotation.sample,
        'Type': annotation.symbol
    })
    
    annotations['AAMI'] = annotations['Type'].map(aami_mapping)
    annotations.insert(0, 'Record', record_name)
    
    ann_path = os.path.join(output_path, f'{record_name}_annotation.csv')
    annotations.to_csv(ann_path, index=False)
    
    print(f'zakończono przetwarzanie rekordu {record_name}')

zakończono przetwarzanie rekordu 100
zakończono przetwarzanie rekordu 101
zakończono przetwarzanie rekordu 102
zakończono przetwarzanie rekordu 103
zakończono przetwarzanie rekordu 104
zakończono przetwarzanie rekordu 105
zakończono przetwarzanie rekordu 106
zakończono przetwarzanie rekordu 107
zakończono przetwarzanie rekordu 108
zakończono przetwarzanie rekordu 109
zakończono przetwarzanie rekordu 111
zakończono przetwarzanie rekordu 112
zakończono przetwarzanie rekordu 113
zakończono przetwarzanie rekordu 114
zakończono przetwarzanie rekordu 115
zakończono przetwarzanie rekordu 116
zakończono przetwarzanie rekordu 117
zakończono przetwarzanie rekordu 118
zakończono przetwarzanie rekordu 119
zakończono przetwarzanie rekordu 121
zakończono przetwarzanie rekordu 122
zakończono przetwarzanie rekordu 123
zakończono przetwarzanie rekordu 124
zakończono przetwarzanie rekordu 200
zakończono przetwarzanie rekordu 201
zakończono przetwarzanie rekordu 202
zakończono przetwarzanie rekordu 203
z

In [8]:
all_annotations = pd.concat([
    pd.read_csv(os.path.join(output_path, f'{name}_annotation.csv'))
    for name in record_names
], ignore_index=True)

print(all_annotations['AAMI'].value_counts(dropna=False))
print(all_annotations.loc[all_annotations['AAMI'].isna(), 'Type'].value_counts())

AAMI
N      90631
Q       8043
V       7236
NaN     3153
S       2781
F        803
Name: count, dtype: int64
Type
+    1291
~     616
!     472
"     437
x     193
|     132
[       6
]       6
Name: count, dtype: int64
